<a href="https://colab.research.google.com/github/Supieiei/Project1_Group7/blob/kp/Part_1_%E0%B8%9E%E0%B8%B4%E0%B8%8A%E0%B8%8D%E0%B8%98%E0%B8%B4%E0%B8%94%E0%B8%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# พิชญธิดา ขัตติยะ 673020638-7

---
# Part 1: Social Listening Warm-up
*งานเดี่ยว — ทุกคนต้องทำและส่งของตนเอง ไม่ต้องนำเสนอ (10 คะแนน)*

ใช้ข้อมูลโพสต์สาธารณะจาก **Bluesky**  
https://huggingface.co/datasets/alpindale/two-million-bluesky-posts

ใช้เพียง **1–3 ไฟล์** ก็พอ ไม่จำเป็นต้องโหลดทั้ง dataset

ฟิลด์ที่มี เช่น `text`, `created_at`, `author`, `uri`, `has_images`, `reply_to`, `predicted_language`

## ตอบ 4 คำถามหลัก

1. **คนโพสต์เมื่อไร:** จำนวนโพสต์ ช่วงเวลาที่ข้อมูลครอบคลุม และชั่วโมงที่โพสต์มากที่สุด
2. **ใช้ภาษาอะไร:** รายงานภาษาหลักและสัดส่วน เพื่อบอกว่าถ้าทำคอนเทนต์ควรรองรับภาษาใด
3. **คนพูดถึงอะไร:** หา hashtag หรือคำ/วลีเด่นที่น่าสนใจ และยกตัวอย่างข้อความจริงประกอบ
4. **ข้อมูลนี้บอกอะไรได้และบอกอะไรไม่ได้:** สร้าง visualization 1 รูป แล้วเขียนข้อจำกัด 3–5 บรรทัด

> Part 1 เป็น warm-up ไม่ต้องทำ influencer analysis หรือ reply-network เว้นแต่สนใจทำเพิ่มเอง

In [ ]:
# ----------------- Your code here -----------------
# ใบ้: อ่าน JSON lines ทีละบรรทัดด้วย generator ไม่ต้องโหลดทั้งไฟล์
# def iter_jsonl(path):
#     with open(path, encoding='utf-8') as f:
#         for line in f:
#             yield json.loads(line)

In [ ]:
!pip install langdetect -q
import pandas as pd
import json
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from langdetect import DetectorFactory, detect

In [ ]:
def iter_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

# สร้าง List เก็บพาธของทั้ง 2 ไฟล์
file_paths = ["posts_20241127_120237.jsonl", "posts_20241127_143458.jsonl"]

# 1. รวบรวมข้อมูลจากทุกไฟล์เข้า List
all_rows = []
for path in file_paths:
    for data in iter_jsonl(path):
        all_rows.append(data)

# 2. แปลงเป็น DataFrame (ตาราง)
df = pd.DataFrame(all_rows)
df

In [ ]:
df.columns

## 1.**คนโพสต์เมื่อไร**: จำนวนโพสต์ ช่วงเวลาที่ข้อมูลครอบคลุม และชั่วโมงที่โพสต์มากที่สุด

In [ ]:
for data in iter_jsonl("posts_20241127_120237.jsonl"):
    print("เวลาที่โพสต์:", data.get("created_at"))

In [ ]:
all_data = [
    data
    for path in file_paths
    for data in iter_jsonl(path)
    if "created_at" in data]
df = pd.DataFrame(all_data)

# 2. แปลงตัวเลขเป็น 'X' เพื่อดูโครงสร้างของข้อความ (Format Pattern)
patterns = df["created_at"].astype(str).str.replace(r"\d", "X", regex=True)
format_counts = patterns.value_counts()

# 3. แสดงผลการตรวจสอบ
print(f"=== ผลการตรวจเช็ค Format (รวมทั้งหมด {len(df):,} รายการ) ===")
print(f"พบ Format ทั้งหมด: {len(format_counts)} รูปแบบ\n")

print("จำนวนของแต่ละ Format:")
for fmt, count in format_counts.items():
    print(f"  - โครงสร้าง `{fmt}` : {count:,} รายการ")

# 4. คำนวณจำนวนที่ไม่เหมือน Format หลัก
main_format = format_counts.index[0]
main_count = format_counts.iloc[0]
different_count = len(df) - main_count

In [ ]:
# อ่านและแปลงข้อมูลด้วยโค้ดที่ยืดหยุ่น (format='ISO8601')
# เพิ่ม errors='coerce' เพื่อเปลี่ยนข้อมูลที่เสียจริงๆ ให้กลายเป็น NaT (Not a Time)
df["created_at"] = pd.to_datetime(df["created_at"], format="ISO8601", errors="coerce")

# เช็คตัวเลขจำนวนข้อมูลและค่าผิดพลาด
valid_count = df["created_at"].notna().sum()
invalid_count = df["created_at"].isna().sum()

# กรองเอาเฉพาะข้อมูลที่ถูกต้องไปใช้วิเคราะห์ต่อ
df = df.dropna(subset=["created_at"])

print("=== 📊 1. รายงานตรวจสอบตัวเลขจำนวนข้อมูล ===")
print(f"• จำนวนโพสต์ทั้งหมด: {len(df):,} โพสต์")
print(f"• จำนวนโพสต์ที่แปลงเวลาสำเร็จ   : {valid_count:,} โพสต์")
print(f"• จำนวนโพสต์ที่ผิดพลาด/เป็นค่าว่าง : {invalid_count:,} โพสต์")

# เช็คตารางตัวเลขแจกแจงรายชั่วโมง
hourly_df = (
    df["created_at"]
    .dt.hour.value_counts()
    .sort_index()
    .reset_index(name="จำนวนโพสต์"))
hourly_df.columns = ["ชั่วโมง (น.)", "จำนวนโพสต์"]

print("\n=== 📈 2. ตารางแจกแจงจำนวนโพสต์ในแต่ละชั่วโมง ===")
print(hourly_df.to_string(index=False))

peak_hour = int(df["created_at"].dt.hour.value_counts().idxmax())
peak_count = int(df["created_at"].dt.hour.value_counts().max())

print("\n" + "=" * 45)
print("=== 🎯 3. สรุปคำตอบ 3 ข้อ ===")
print(f"1. จำนวนโพสต์ทั้งหมดที่สมบูรณ์: {valid_count:,} โพสต์")
print(f"2. ช่วงเวลาที่ครอบคลุม: ตั้งแต่ {df['created_at'].min()} ถึง {df['created_at'].max()} (UTC)")
print(f"3. ชั่วโมงที่โพสต์มากที่สุด: ช่วงเวลา {peak_hour:02d}:00 น. - {peak_hour:02d}:59 น. (จำนวน {peak_count:,} โพสต์)")

## 2.**ใช้ภาษาอะไร**: รายงานภาษาหลักและสัดส่วน เพื่อบอกว่าถ้าทำคอนเทนต์ควรรองรับภาษาใด

In [ ]:
# ตั้งค่าให้ผลการวิเคราะห์ภาษาเสถียร
DetectorFactory.seed = 0

# ฟังก์ชันตรวจจับภาษาพร้อมจัดการ Error (กรณีข้อความสั้นเกินไปหรือเป็นอิโมจิล้วน)
def detect_language(text):
    if not isinstance(text, str) or len(text.strip()) < 3:
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"

print("⏳ กำลังวิเคราะห์ภาษาจากข้อความ...")
df["language"] = df["text"].apply(detect_language)

# 5. สรุปตารางสัดส่วน (Percentage)
summary_df = df["language"].value_counts().reset_index()
summary_df.columns = ["รหัสภาษา (Language Code)", "จำนวนโพสต์"]
summary_df["สัดส่วน (%)"] = (
    summary_df["จำนวนโพสต์"] / len(df) * 100).round(2)

print("\n=== 📊 1. สัดส่วนภาษาทั้งหมดในข้อมูล ===")
print(summary_df.head(10).to_string(index=False))

⏳ กำลังวิเคราะห์ภาษาจากข้อความ...


## 3.**คนพูดถึงอะไร**: หา hashtag หรือคำ/วลีเด่นที่น่าสนใจ และยกตัวอย่างข้อความจริงประกอบ